In [ ]:
# 이미지를 읽고 설명하기
!pip install openai python-dotenv

In [ ]:
# 단일 실행형 스크립 -------------
from openai import OpenAI
import io, os
import base64
from PIL import Image
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)
model = "gpt-4o-mini"

# 프롬프트 작성
system_prompt = "당신은 그림을 보고 내용에 대한 설명을 잘하는 전문가입니다."   # 모델의 역할 지정
user_prompt = "다음 이미지를 보고 무슨 내용인지 작성하시오"    # 실제 사용자 요청 내용

# 이미지 열기
image_path = "pic.jpeg"
with open(image_path, 'rb') as f:
    image_data = f.read()
    image = Image.open(io.BytesIO(image_data)).convert("RGB")
    print(image.size)   # (658, 465)

# 이미지를 base64 문자열로 변환 - OpenAI api가 원함
base64_image = base64.b64encode(image_data).decode("utf-8")
# print(base64_image)


In [ ]:
# 멀티 모달(텍스트 + 이미지) 질의 실행
response = client.responses.create(
    model=model,
    input = [
        {   # 첫번째 전달할 메세지 : system 역할 메세지
            "role":"system",   # 모델의 역할, 태도, 규칙, 답변형식 등 지정
            "content":[
                {
                    "type":"input_text",
                    "text":system_prompt
                }
            ]
        },
        {   # 두번째 전달할 메세지 : user 역할 메세지
            "role":"user",    # 사용자의 실제 질문, 요청, 이미지, 텍스트 전달...
            "content":[
                {
                    "type":"input_text",
                    "text":user_prompt
                },
                {
                    "type":"input_image",
                    "image_url":f"data:image/jpeg;base64,{base64_image}"
                }
            ]
        }
    ]
)

# print(response)
print('이미지 설명 : ')
if response.output_text:
    print(response.output_text)
else:
    print("응답이 비었어요")


In [ ]:
# 클래스 기반 구조화 -------------
class MyMultiModel:
    def __init__(self, client, model, system_prompt="", user_prompt=""):
        self.client = client
        self.model = model
        self.system_prompt = system_prompt
        self.user_prompt = user_prompt

    # 이미지 경로를 받아 스트리밍 응답을 생성
    def streamMethod(self, image_path):
        with open(image_path, 'rb') as f:
            image_bytes = f.read()

        base64_image = base64.b64encode(image_bytes).decode("utf-8")

        response = self.client.responses.create(
            model=self.model,
            input = [
                {   # 첫번째 전달할 메세지 : system 역할 메세지
                    "role":"system",   # 모델의 역할, 태도, 규칙, 답변형식 등 지정
                    "content":[
                        {
                            "type":"input_text",
                            "text":self.system_prompt
                        }
                    ]
                },
                {   # 두번째 전달할 메세지 : user 역할 메세지
                    "role":"user",    # 사용자의 실제 질문, 요청, 이미지, 텍스트 전달...
                    "content":[
                        {
                            "type":"input_text",
                            "text":self.user_prompt
                        },
                        {
                            "type":"input_image",
                            "image_url":f"data:image/jpeg;base64,{base64_image}"
                        }
                    ]
                }
            ],
            stream=True    # 스트리밍 응답으로 생성되는 텍스트가 한 줄씩 실시간 출력됨
        )
        return response


# 스트리밍 응답 출력 함수
def stream_responseFunc(response):
    for event in response:  # 스트리밍으로 전달되는 이벤트를 하나씩 수행

        # delta : 스트리밍 중에 조금씩 도착하는 '응답 조각'
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)

llm = "gpt-4o-mini"
system_prompt = "당신은 시인입니다. 당신의 임무는 주어진 이미지를 가지고 시를 작성하는 것입니다"   # 모델의 역할 지정
user_prompt = "다음 이미지를 보고 무슨 시를 작성하시오"    # 실제 사용자 요청 내용

multimodal_gpt = MyMultiModel(
    client,
    model=model,
    system_prompt=system_prompt,
    user_prompt=user_prompt
)

IMAGE_PATH = "pic.jpeg"
response = multimodal_gpt.streamMethod(IMAGE_PATH)
stream_responseFunc(response)
